# Block 05 – Pause, look, resume

**You'll see:** the run pause at `interrupt()`, the payload a human would review, resuming with approve or deny, and the node running twice.

The pause and the resume are in **separate cells** on purpose: in real life they may be hours apart.

In [ ]:
# Setup: run this cell first. It finds the repo, makes it importable and loads .env
import os, sys, pathlib

def find_repo_root():
    starts = []
    if "__vsc_ipynb_file__" in globals():                 # VS Code tells us where this notebook is
        starts.append(pathlib.Path(globals()["__vsc_ipynb_file__"]).resolve().parent)
    starts.append(pathlib.Path.cwd().resolve())
    for start in starts:
        for folder in [start, *start.parents]:
            if (folder / "common" / "config.py").exists():
                return folder
            if (folder / "agents-weekend" / "common" / "config.py").exists():
                return folder / "agents-weekend"
    raise RuntimeError(f"Can't find the agents-weekend repo from {starts}. "
                       "Open the agents-weekend folder in VS Code, or set ROOT by hand.")

ROOT = find_repo_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

from common.config import get_llm, get_cheap_llm, DEFAULT_MODEL, CHEAP_MODEL
from common.tracing import langfuse_config, trace_attributes, flush, tracing_enabled
print("main model:", DEFAULT_MODEL, "| cheap model:", CHEAP_MODEL, "| Langfuse:", tracing_enabled())

## 1. Build the account agent with a checkpointer

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from common.loader import load_block
b05 = load_block("05-human-in-the-loop")
graph = b05.build_graph(get_llm(), checkpointer=InMemorySaver())
config = langfuse_config(thread_id="bob-reset-notebook", recursion_limit=15)

## 2. Ask for a reset → the run pauses

In [ ]:
with trace_attributes(session_id="bob-reset-notebook", tags=["notebook-05"]):
    result = graph.invoke({"messages": [("user", "Hi, this is bob. I'm locked out, please reset my password.")]},
                          config=config)
print("paused?", "__interrupt__" in result)
if "__interrupt__" in result:
    print("payload for the human:", result["__interrupt__"][0].value)
print("graph will continue at:", graph.get_state(config).next)

## 3. The human approves → resume with the same thread_id
Change `True` to `False` (and run cell 2 again first) to see a denial.

In [ ]:
with trace_attributes(session_id="bob-reset-notebook", tags=["notebook-05"]):
    result = graph.invoke(Command(resume={"approved": True, "note": ""}), config=config)
print("paused?", "__interrupt__" in result)
print(result["messages"][-1].text)

## 4. Proof that the node re-runs from the top on resume
A counter in a test tool goes up twice: once before the pause, once on resume.

In [ ]:
from langchain_core.tools import tool
from langgraph.types import interrupt
from common.graphs import build_tool_agent

runs = []

@tool
def risky_action(target: str) -> str:
    """A sensitive action that needs human approval. Use it when asked to run the risky action."""
    runs.append("code before interrupt ran")      # imagine an email or API call here
    answer = interrupt({"question": f"Run risky action on {target}?"})
    return "done" if answer else "denied"

g = build_tool_agent(get_llm(), [risky_action], "Use risky_action when the user asks.", checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "rerun-demo"}}
g.invoke({"messages": [("user", "Please run the risky action on server-1")]}, cfg)
print("after pause: ", len(runs), "run(s)")
g.invoke(Command(resume=True), cfg)
print("after resume:", len(runs), "run(s)  <- the node ran again from the top")

In [ ]:
flush()